# Section 1 — Setup & Imports

In [1]:
# =========================
# Section 1 — Setup & Imports
# =========================
import os
import json
import re
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

# Section 2 — Document Loading: PDFs

In [2]:
# =========================
# Section 2 — Document Loading: PDFs
# =========================
from langchain.document_loaders import PyPDFLoader

pdf_path = "./docs/MachineLearning-Lecture01.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Total PDF pages:", len(pages))
print("\nPreview:\n", pages[0].page_content[:500])
print("\nMetadata:\n", pages[0].metadata)

Total PDF pages: 22

Preview:
 MachineLearning-Lecture01  
Instructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine 
learning class. So what I wanna do today is just spend a little time going over the logistics 
of the class, and then we'll start to talk a bit about machine learning.  
By way of introduction, my name's Andrew Ng and I'll be instructor for this class. And so 
I personally work in machine learning, and I've worked on it for about 15 years now, and 
I actually think that machine learning is the 

Metadata:
 {'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'author': '', 'moddate': '2008-07-11T11:25:23-07:00', 'title': '', 'source': './docs/MachineLearning-Lecture01.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}


# Section 3 — Document Loading: YouTube (Online-First with Local Fallback)

# Document Loading

In [3]:
# =========================
# Section 3 — Document Loading: YouTube (Online-First with Local Fallback)
# =========================
from langchain_community.document_loaders import YoutubeLoader, TextLoader
from langchain.schema import Document

url = "https://www.youtube.com/watch?v=jGwO_UgTS7I"
save_dir = Path("./docs/youtube")
save_dir.mkdir(parents=True, exist_ok=True)

def extract_video_id(youtube_url: str) -> str:
    m = re.search(r"(?:v=|be/)([A-Za-z0-9_-]{11})", youtube_url)
    if not m:
        raise ValueError(f"Could not extract video id from URL: {youtube_url}")
    return m.group(1)

vid = extract_video_id(url)
txt_path = save_dir / f"{vid}.transcript.txt"
json_path = save_dir / f"{vid}.transcript.json"

def load_from_local() -> list[Document] | None:
    if txt_path.exists():
        return TextLoader(str(txt_path)).load()
    if json_path.exists():
        data = json.loads(json_path.read_text(encoding="utf-8"))
        joined = " ".join(seg.get("text", "") for seg in data)
        return [Document(page_content=joined, metadata={"source": str(json_path), "video_id": vid})]
    return None

docs = None

# 3.1 Try online fetch (works locally; may be blocked in cloud VMs)
try:
    y_loader = YoutubeLoader.from_youtube_url(url, add_video_info=False)
    docs = y_loader.load()
    # Cache a clean text transcript for future runs
    cleaned_text = docs[0].page_content if docs else ""
    txt_path.write_text(cleaned_text, encoding="utf-8")
    print("Fetched transcript online and cached to:", txt_path)
except Exception as e:
    print(f"Online transcript fetch failed ({type(e).__name__}): {e}")
    print("Attempting to load a pre-downloaded local transcript...")

# 3.2 Fallback to local cache
if docs is None:
    local_docs = load_from_local()
    if local_docs:
        docs = local_docs
        print("Loaded transcript from local cache:", txt_path if txt_path.exists() else json_path)
    else:
        raise RuntimeError(
            "Could not load YouTube transcript.\n"
            "- If running in the cloud, YouTube may block the VM.\n"
            "- Fix by running locally once (to cache) or place a transcript at:\n"
            f"  {txt_path}  (plain text)  OR  {json_path}  (list of segments)."
        )

print("\nYouTube transcript preview:\n", docs[0].page_content[:500])


Online transcript fetch failed (RequestBlocked): 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=jGwO_UgTS7I! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the 

## Section 4 — Document Loading: URLs (Web Pages)

In [4]:
# =========================
# Section 4 — Document Loading: URLs (Web Pages)
# =========================
import os
os.environ['USER_AGENT'] = 'myagent'

from langchain.document_loaders import WebBaseLoader

web_url = "https://python.langchain.com/v0.2/docs/integrations/document_loaders/youtube_audio/"
w_loader = WebBaseLoader(web_url)
web_docs = w_loader.load()

print("Web page preview:\n", web_docs[0].page_content[:500].strip())

Web page preview:
 YouTube audio | 🦜️🔗 LangChain







Skip to main contentA newer LangChain version is out! Check out the latest version.IntegrationsAPI referenceLatestLegacyMorePeopleContributingCookbooks3rd party tutorialsYouTubearXivv0.2Latestv0.2v0.1🦜️🔗LangSmithLangSmith DocsLangChain HubJS/TS Docs💬SearchProvidersProvidersAnthropicAWSGoogleHugging FaceMicrosoftOpenAIMoreComponentsChat modelsChat modelsAI21 LabsAlibaba Cloud PAI EASAnthropic[Deprecated] Experimental Anthropic Tools WrapperAnyscaleAzure O


## Section 5 — Document Loading: Notion Directory

In [5]:
# =========================
# Section 5 — Document Loading: Notion Directory
# =========================
from langchain.document_loaders import NotionDirectoryLoader

notion_dir = "./docs/Notion_DB"
n_loader = NotionDirectoryLoader(notion_dir)
notion_docs = n_loader.load()

print("Notion doc preview:\n", notion_docs[0].page_content[:200])
print("\nNotion metadata:\n", notion_docs[0].metadata)

Notion doc preview:
 # Your check-list

To give you an idea of how and when to approach all the different topics, here's a checklist with a timeline. You can duplicate this page for your own use.

**Daily**

- [ ]  Give c

Notion metadata:
 {'source': 'docs/Notion_DB/Your check-list 07455ff3b1364229bfc1ae3462e58030.md'}


## Document Loading

## PDFs

In [6]:
import os
import openai
from dotenv import load_dotenv
from langchain.document_loaders import PyPDFLoader

load_dotenv()

openai.api_key=os.getenv("OPENAI_API_KEY")

In [7]:
loader=PyPDFLoader("./docs/MachineLearning-Lecture01.pdf")
pages=loader.load()

In [8]:
len(pages)
print(len(pages))

22


In [9]:
page=pages[0]

In [10]:
print(page.page_content[0:500])

MachineLearning-Lecture01  
Instructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine 
learning class. So what I wanna do today is just spend a little time going over the logistics 
of the class, and then we'll start to talk a bit about machine learning.  
By way of introduction, my name's Andrew Ng and I'll be instructor for this class. And so 
I personally work in machine learning, and I've worked on it for about 15 years now, and 
I actually think that machine learning is the 


In [11]:
page.metadata
print(pages[0].metadata)

{'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2008-07-11T11:25:23-07:00', 'author': '', 'moddate': '2008-07-11T11:25:23-07:00', 'title': '', 'source': './docs/MachineLearning-Lecture01.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}


# YouTube

In [14]:
from langchain_community.document_loaders import YoutubeLoader, TextLoader
from langchain.schema import Document

In [15]:
url = "https://www.youtube.com/watch?v=jGwO_UgTS7I"
save_dir = Path("./docs/youtube")
save_dir.mkdir(parents=True, exist_ok=True)

def extract_video_id(youtube_url: str) -> str:
    m = re.search(r"(?:v=|be/)([A-Za-z0-9_-]{11})", youtube_url)
    if not m:
        raise ValueError(f"Could not extract video id from URL: {youtube_url}")
    return m.group(1)

video_id = extract_video_id(url)
txt_path = save_dir / f"{video_id}.transcript.txt"     # plain-text cache your notebook expects
json_path = save_dir / f"{video_id}.transcript.json"   # optional segments cache

def load_from_local():
    # Prefer TXT cache
    if txt_path.exists():
        return TextLoader(str(txt_path)).load()
    # Optional JSON -> join segments
    if json_path.exists():
        data = json.loads(json_path.read_text(encoding="utf-8"))
        joined = " ".join(seg.get("text", "") for seg in data)
        return [Document(page_content=joined, metadata={"source": str(json_path), "video_id": video_id})]
    return None

docs = None

# 1) Try normal online fetch (works on laptops; often blocked in cloud VMs)
try:
    y_loader = YoutubeLoader.from_youtube_url(url, add_video_info=False)
    docs = y_loader.load()
    # Cache plain text if available
    if docs and docs[0].page_content:
        txt_path.write_text(docs[0].page_content, encoding="utf-8")
        print("Fetched transcript online and cached:", txt_path)
except Exception as e:
    print(f"Online fetch blocked/failed: {type(e).__name__}: {e}")

# 2) Fallback to local cache if online failed
if docs is None:
    local_docs = load_from_local()
    if local_docs:
        docs = local_docs
        print("Loaded transcript from local cache:", txt_path if txt_path.exists() else json_path)
    else:
        # 3) OPTIONAL: try direct youtube-transcript-api with a proxy (if provided)
        proxy_url = os.getenv("HTTPS_PROXY") or os.getenv("HTTP_PROXY")
        if proxy_url:
            try:
                from youtube_transcript_api import YouTubeTranscriptApi
                segments = YouTubeTranscriptApi.get_transcript(
                    video_id, languages=["en"],
                    proxies={"http": proxy_url, "https": proxy_url}
                )
                text = " ".join(seg["text"] for seg in segments)
                # cache and return as Document
                txt_path.write_text(text, encoding="utf-8")
                docs = [Document(page_content=text, metadata={"source": str(txt_path), "video_id": video_id})]
                print("Fetched via proxy and cached:", txt_path)
            except Exception as e:
                raise RuntimeError(
                    "Could not load YouTube transcript (blocked online, no local cache, proxy attempt failed).\n"
                    "Fix by seeding a transcript file at:\n"
                    f"  {txt_path}\n"
                    "You can create it from VTT/SRT with yt-dlp (locally) and copy it here."
                ) from e
        else:
            raise RuntimeError(
                "Could not load YouTube transcript (blocked online and no local cache).\n"
                "Fix in one of these ways:\n"
                "  A) Run locally once to auto-cache the transcript, or\n"
                "  B) Create the file manually at:\n"
                f"     {txt_path}\n"
                "     (Use yt-dlp to download captions, then convert to plain text.)\n"
                "  C) Set HTTPS_PROXY/HTTP_PROXY and rerun."
            )

# Preview
print(docs[0].page_content[:500])

Online fetch blocked/failed: RequestBlocked: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=jGwO_UgTS7I! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the acco

In [16]:
docs[0].page_content[0:500]

'welcome to 69 machine learning some of welcome to 69 machine learning some of welcome to 69 machine learning some of you know that this closet or the you know that this closet or the you know that this closet or the Stanford for a long time and this is Stanford for a long time and this is Stanford for a long time and this is often the cost that I most look forward often the cost that I most look forward often the cost that I most look forward to teaching each year because this is to teaching eac'

## URLs

In [17]:
import os 
os.environ['USER_AGENT'] = 'myagent'
from langchain.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/integrations/document_loaders/youtube_audio/")

In [18]:
docs=loader.load()

docs[0].page_content[0:500].strip()

'YouTube audio | 🦜️🔗 LangChain\n\n\n\n\n\n\n\nSkip to main contentA newer LangChain version is out! Check out the latest version.IntegrationsAPI referenceLatestLegacyMorePeopleContributingCookbooks3rd party tutorialsYouTubearXivv0.2Latestv0.2v0.1🦜️🔗LangSmithLangSmith DocsLangChain HubJS/TS Docs💬SearchProvidersProvidersAnthropicAWSGoogleHugging FaceMicrosoftOpenAIMoreComponentsChat modelsChat modelsAI21 LabsAlibaba Cloud PAI EASAnthropic[Deprecated] Experimental Anthropic Tools WrapperAnyscaleAzure O'

## Notion DB

In [19]:
from langchain.document_loaders import NotionDirectoryLoader
loader = NotionDirectoryLoader("./docs/Notion_DB")
docs=loader.load()

In [20]:
print(docs[0].page_content[0:200])

# Your check-list

To give you an idea of how and when to approach all the different topics, here's a checklist with a timeline. You can duplicate this page for your own use.

**Daily**

- [ ]  Give c


In [21]:
docs[0].metadata

{'source': 'docs/Notion_DB/Your check-list 07455ff3b1364229bfc1ae3462e58030.md'}